# 06 · Inter-Annotator Agreement (Addressee & Addressee Type)

Measures annotation agreement on two variables: **addressee** (the specific entity being addressed) and **addressee_type** (category: individual / group / monologue / non_human / unclear).

**Two tests**
1. **Test 1 — Human vs Human (Cohen's κ, pairwise)**: the 3 annotators (majo / sonia / oceane) used an interleaved design where each film was annotated by **2 of them**. Because the rater pair is not fixed, we use **pairwise Cohen's kappa** (Fleiss is not applicable — it assumes a fixed/random rater set), then take a sample-size-weighted average.
2. **Test 2 — Human (gold) vs LLM (Cohen's κ)**: gold = rows where the two annotators **agree**; we also report LLM vs each individual annotator.

**Method notes**
- Each κ is reported alongside the **raw observed agreement (%agree)**: the categories are heavily dominated by `individual`, and under such imbalance κ is suppressed by the "kappa paradox", so it must be read together with %agree.
- Both variables are nominal → **unweighted** Cohen's κ; 95% CI from 1000 bootstrap resamples.
- **Excluded films (3)**: `soul_2020` (script issues), `toy_story_3_2010` (no annotations), `raya_and_the_last_dragon_2021` (only 2 single-rater rows from sonia, no pair). → 18 − 3 = **15 films** analyzed.

**Data cleaning**
- Normalize to lowercase / strip / `-`→`_` (e.g. `non-human`→`non_human`).
- addressee_type dirty-value mapping: `unknow`/`unknown`/`other`→`unclear`; entity names leaked into the type column (type==addressee and non-standard) → `individual`.
- Non-informative tokens in the addressee column (`unknow`/`unknown`/`other`) → missing.

**Rule-based fill for majo's addressee_type**
majo left addressee_type blank on **40 rows** (mulan / up / zootopia) where she only filled addressee. Her addressee already encodes the type: a single entity written as `film_charactername` → `individual`; a type word written directly (`non_human`/`monologue`) → that type. We fill from this rule (cross-check: among rows derived as individual, 100% of those sonia also labeled are individual), which brings **mulan** etc. into the addressee_type analysis.

> Landis & Koch (1977) κ scale: <0 poor · 0–.20 slight · .21–.40 fair · .41–.60 moderate · .61–.80 substantial · .81–1 almost perfect

In [1]:
import pandas as pd, numpy as np, re
from sklearn.metrics import cohen_kappa_score

TYPE_TOKENS  = {"individual","group","monologue","non_human","unclear"}
NONINFO      = {"unknow","unknown","other"}   # non-informative addressee tokens ('error' only occurred in soul, already excluded)
# Excluded: soul (script issues) · toy_story_3 (no annotations) · raya (only 2 single-rater rows, no pair)
EXCLUDE_FILMS= {"soul_2020","toy_story_3_2010","raya_and_the_last_dragon_2021"}
ANN          = ["majo","sonia","oceane"]
PAIRS        = [("majo","sonia"),("majo","oceane"),("sonia","oceane")]

def norm(x):
    if pd.isna(x): return np.nan
    return str(x).strip().lower().replace("-","_")

In [2]:
# Load both datasets and align on utterance_id
a = pd.read_excel("../data/03_validation_dataset/addressee_validation_results/data_for_annotation.xlsx", sheet_name="validation_for_annotation")
p = pd.read_excel("../data/03_validation_dataset/addressee_validation_results/data_for_prediction.xlsx")[["utterance_id","addressee_llm","addressee_type_llm"]]
df = a.merge(p, on="utterance_id", how="left")
df = df[~df.film_id.isin(EXCLUDE_FILMS)].copy()
print("Rows analyzed:", len(df), "| Films:", df.film_id.nunique())

Rows analyzed: 225 | Films: 15


### Cleaning

In [3]:
def clean_type(t, addr):
    t = norm(t)
    if t is np.nan: return np.nan
    if t in TYPE_TOKENS: return t
    if t in {"unknow","unknown","other"}: return "unclear"
    if t == norm(addr): return "individual"   # entity name leaked into the type column
    return np.nan

def clean_addr(x):
    x = norm(x)
    return np.nan if x in NONINFO else x

for w in ANN:
    df["addr_"+w] = df["addressee_"+w].map(clean_addr)
    df["type_"+w] = df.apply(lambda r: clean_type(r["addressee_type_"+w], r["addressee_"+w]), axis=1)

df["addr_llm"] = df["addressee_llm"].map(clean_addr)
df["type_llm"] = df["addressee_type_llm"].map(norm)

### Rule-based fill for majo's addressee_type (brings mulan in)

In [4]:
def derive_majo_type(addr):
    v = norm(addr)
    if v is np.nan: return np.nan
    if v in TYPE_TOKENS: return v
    if v in {"unknow","unknown","other"}: return "unclear"
    if re.match(r"^[a-z0-9]+(_[a-z0-9]+)+$", v): return "individual"  # film_charactername -> single entity
    return np.nan

mask = df["type_majo"].isna() & df["addressee_majo"].notna()
df.loc[mask, "type_majo"] = df.loc[mask, "addressee_majo"].map(derive_majo_type)
print(f"Rule-filled majo addressee_type: {mask.sum()} rows")
print("Filled distribution:", df.loc[mask,"type_majo"].value_counts().to_dict())

Rule-filled majo addressee_type: 40 rows
Filled distribution: {'individual': 37, 'non_human': 2, 'monologue': 1}


### κ helper (Cohen's κ + %agree + bootstrap 95% CI)

In [5]:
def kappa_block(d, c1, c2, seed=0, B=1000):
    sub = d[d[c1].notna() & d[c2].notna()]
    n = len(sub)
    if n < 2: return dict(n=n, kappa=np.nan, agree=np.nan, lo=np.nan, hi=np.nan)
    x, y = sub[c1].astype(str).values, sub[c2].astype(str).values
    k = cohen_kappa_score(x, y)
    agree = (x == y).mean()
    rng = np.random.default_rng(seed); ks=[]
    for _ in range(B):
        b = rng.integers(0, n, n)
        try: ks.append(cohen_kappa_score(x[b], y[b]))
        except Exception: pass
    lo, hi = (np.nanpercentile(ks,[2.5,97.5]) if ks else (np.nan,np.nan))
    return dict(n=n, kappa=k, agree=agree, lo=lo, hi=hi)

## Test 1 — Human vs Human (pairwise Cohen's κ)

In [6]:
rows=[]
for var,label in [("addr","addressee"),("type","addressee_type")]:
    tot_n=0; wsum=0
    for w1,w2 in PAIRS:
        r = kappa_block(df, f"{var}_{w1}", f"{var}_{w2}")
        rows.append(dict(variable=label, pair=f"{w1}–{w2}", n=r["n"],
                         pct_agree=r["agree"], kappa=r["kappa"],
                         ci=f"[{r['lo']:.2f}, {r['hi']:.2f}]" if r["n"]>=2 else ""))
        if r["n"]>=2: tot_n+=r["n"]; wsum+=r["kappa"]*r["n"]
    rows.append(dict(variable=label, pair="weighted avg", n=tot_n,
                     pct_agree=np.nan, kappa=wsum/tot_n if tot_n else np.nan, ci=""))
t1 = pd.DataFrame(rows)
t1_show = t1.copy()
t1_show["pct_agree"]=t1_show["pct_agree"].map(lambda v: f"{v:.0%}" if pd.notna(v) else "")
t1_show["kappa"]=t1_show["kappa"].map(lambda v: f"{v:.3f}" if pd.notna(v) else "")
t1_show

,variable,pair,n,pct_agree,kappa,ci
0,addressee,majo–sonia,100,90%,0.895,"[0.83, 0.95]"
1,addressee,majo–oceane,87,71%,0.701,"[0.60, 0.79]"
2,addressee,sonia–oceane,33,70%,0.659,"[0.48, 0.83]"
3,addressee,weighted avg,220,,0.783,
4,addressee_type,majo–sonia,100,93%,0.796,"[0.65, 0.92]"
5,addressee_type,majo–oceane,90,80%,0.361,"[0.14, 0.57]"
6,addressee_type,sonia–oceane,32,75%,0.478,"[0.19, 0.74]"
7,addressee_type,weighted avg,222,,0.573,


### addressee_type: per-film overlap counts (transparency — confirms mulan is included)

In [7]:
ov=[]
for fid,g in df.groupby("film_id"):
    for w1,w2 in PAIRS:
        n=(g[f"type_{w1}"].notna()&g[f"type_{w2}"].notna()).sum()
        if n>0: ov.append(dict(film=fid, pair=f"{w1}–{w2}", n_overlap=n))
pd.DataFrame(ov)

,film,pair,n_overlap
0,beautyandthebeast_1991,majo–sonia,3
1,beautyandthebeast_1991,majo–oceane,15
2,beautyandthebeast_1991,sonia–oceane,3
3,cars2,majo–oceane,15
4,coco_2017,majo–oceane,15
5,elemental_2023,majo–oceane,15
6,encanto_2021,majo–sonia,15
7,findingnemo,majo–sonia,15
8,frozen_2013,majo–sonia,15
9,incredibles_2_2018,majo–sonia,15


## Error analysis — where oceane diverges (addressee_type)

Diagnostic for the lower oceane pairings: label-usage distribution per rater, confusion matrices of oceane against each partner, and a row-level disagreement table.

In [8]:
# Label-usage distribution per rater (addressee_type, % of labeled rows)
dist = pd.DataFrame({w: df["type_"+w].value_counts(normalize=True) for w in ANN})
dist["llm"] = df["type_llm"].value_counts(normalize=True)
(dist.fillna(0) * 100).round(0).astype(int)

,majo,sonia,oceane,llm
group,13,20,10,17
individual,81,73,80,76
monologue,4,4,7,3
non_human,1,3,1,3
unclear,1,0,3,1


In [9]:
# Confusion matrices on addressee_type: oceane vs each partner (rows = partner, cols = oceane)
for partner in ["majo","sonia"]:
    sub = df[df["type_"+partner].notna() & df["type_oceane"].notna()]
    nd  = int((sub["type_"+partner] != sub["type_oceane"]).sum())
    print(f"{partner} (rows) vs oceane (cols)  —  n={len(sub)}, disagreements={nd}")
    display(pd.crosstab(sub["type_"+partner], sub["type_oceane"]))

majo (rows) vs oceane (cols)  —  n=90, disagreements=18


type_oceane,group,individual,monologue,non_human,unclear
type_majo,,,,,
group,3,5,0,0,0
individual,4,66,0,1,3
monologue,0,3,3,0,0
non_human,0,0,1,0,0
unclear,0,0,1,0,0


sonia (rows) vs oceane (cols)  —  n=32, disagreements=8


type_oceane,group,individual,monologue
type_sonia,,,
group,3,5,1
individual,1,19,0
monologue,1,0,2


In [10]:
# Row-level disagreements involving oceane (addressee_type)
detail=[]
for partner in ["majo","sonia"]:
    sub = df[df["type_"+partner].notna() & df["type_oceane"].notna()]
    dis = sub[sub["type_"+partner] != sub["type_oceane"]]
    for _,r in dis.iterrows():
        detail.append(dict(film=r.film_id, speaker=r.speaker, partner=partner,
                           partner_type=r["type_"+partner], oceane_type=r["type_oceane"],
                           llm_type=r["type_llm"]))
detail_df = pd.DataFrame(detail)
print(f"Total disagreements involving oceane (addressee_type): {len(detail_df)}")
detail_df

Total disagreements involving oceane (addressee_type): 26


,film,speaker,partner,partner_type,oceane_type,llm_type
0,beautyandthebeast_1991,TEAPOT,majo,group,individual,group
1,beautyandthebeast_1991,MANTLECLOCK,majo,individual,unclear,non_human
2,cars2,MATER,majo,individual,group,individual
3,cars2,MCQUEEN,majo,individual,group,individual
4,coco_2017,T O OSCAR,majo,group,individual,individual
5,coco_2017,MAM IMELDA,majo,group,individual,individual
6,coco_2017,H CTOR,majo,group,individual,individual
7,elemental_2023,TEENAGE EMBER,majo,individual,unclear,individual
8,elemental_2023,EMBER,majo,monologue,individual,unclear
9,up,CARL,majo,individual,group,individual


**Findings — why oceane's addressee_type κ is lower**

The lower κ is *not* a sign of poor annotation quality; it comes from three overlapping sources, amplified by the kappa paradox:

1. **group ↔ individual boundary.** oceane draws the "addressing a crowd vs one person" line differently (e.g. cars2 Mater/McQueen: oceane=`group`, majo=`individual`). It goes both ways — in coco oceane labels `individual` while majo labels `group`, and there oceane actually matches the LLM. So this is genuine task ambiguity, not one-sided error.
2. **More `unclear` / blanks.** oceane uses `unclear` ~3% (others ~0–1%) and sometimes leaves addressee blank; when the partner commits to a concrete label this counts as a disagreement.
3. **monologue / non_human edge cases** (talking to a pet, talking to oneself).

Amplifiers: (a) **kappa paradox** — ~80% of labels are `individual`, so a modest number of boundary disagreements crushes κ even though %agree stays ~80%; (b) **film difficulty** — oceane's assigned films (cars2 / coco / up / zootopia) are ensemble/crowd scenes with inherently more group/individual ambiguity than the two-hander dialogues in the majo–sonia set.

## Test 2 — Human (gold) vs LLM

**gold** = the label where the two annotators on a row **agree** (rows with disagreement, or only one annotator, are not counted as gold). LLM vs each individual annotator is also reported as a robustness check.

In [11]:
def build_gold(d, var):
    cols=[f"{var}_{w}" for w in ANN]
    def g(row):
        vals=[row[c] for c in cols if pd.notna(row[c])]
        return vals[0] if (len(vals)>=2 and len(set(vals))==1) else np.nan
    return d.apply(g, axis=1)

rows=[]
for var,label in [("addr","addressee"),("type","addressee_type")]:
    df["gold"]=build_gold(df, var)
    r=kappa_block(df,"gold",f"{var}_llm")
    rows.append(dict(variable=label, comparison="gold (annotators agree) vs LLM", n=r["n"],
                     pct_agree=r["agree"], kappa=r["kappa"],
                     ci=f"[{r['lo']:.2f}, {r['hi']:.2f}]"))
    for w in ANN:
        rr=kappa_block(df,f"{var}_{w}",f"{var}_llm")
        rows.append(dict(variable=label, comparison=f"{w} vs LLM", n=rr["n"],
                         pct_agree=rr["agree"], kappa=rr["kappa"],
                         ci=f"[{rr['lo']:.2f}, {rr['hi']:.2f}]" if rr["n"]>=2 else ""))
t2=pd.DataFrame(rows)
t2_show=t2.copy()
t2_show["pct_agree"]=t2_show["pct_agree"].map(lambda v: f"{v:.0%}" if pd.notna(v) else "")
t2_show["kappa"]=t2_show["kappa"].map(lambda v: f"{v:.3f}" if pd.notna(v) else "")
t2_show

,variable,comparison,n,pct_agree,kappa,ci
0,addressee,gold (annotators agree) vs LLM,170,85%,0.842,"[0.79, 0.90]"
1,addressee,majo vs LLM,188,78%,0.769,"[0.71, 0.83]"
2,addressee,sonia vs LLM,137,80%,0.791,"[0.72, 0.86]"
3,addressee,oceane vs LLM,117,63%,0.618,"[0.53, 0.70]"
4,addressee_type,gold (annotators agree) vs LLM,183,90%,0.645,"[0.49, 0.77]"
5,addressee_type,majo vs LLM,188,83%,0.508,"[0.37, 0.63]"
6,addressee_type,sonia vs LLM,137,88%,0.724,"[0.61, 0.83]"
7,addressee_type,oceane vs LLM,119,75%,0.330,"[0.15, 0.50]"


## Summary (interpretation)

- **Human–human addressee** agreement is high (weighted κ≈0.78; majo–sonia reaches 0.90). **addressee_type** weighted κ≈0.57; the lower majo–oceane value (κ≈0.36) is driven mainly by the heavy `individual` skew (%agree is still 80% — a textbook kappa paradox).
- **Human–LLM**: addressee gold-vs-LLM κ≈0.84 (substantial~almost perfect); addressee_type κ≈0.65 (substantial). The LLM is closest to sonia and lowest against oceane.
- `n` is the number of valid paired rows for each comparison; read κ together with %agree and the CI.

> Values change as the data is updated — the executed results in this notebook are authoritative.

In [12]:
# Export result tables as CSV
from pathlib import Path
out = Path("../data/03_validation_dataset/addressee_validation_results"); out.mkdir(parents=True, exist_ok=True)

t1.round(3).to_csv(out / "test1_human_human.csv", index=False)    # Test 1: pairwise human-human kappa
t2.round(3).to_csv(out / "test2_human_llm.csv",   index=False)    # Test 2: human gold / per-annotator vs LLM
detail_df.to_csv(out / "test3_human_disagreements.csv", index=False)   # row-level disagreements involving oceane

print("Saved to", out.resolve())
for f in ["test1_human_human.csv", "test2_human_llm.csv", "test3_human_disagreements.csv"]:
    print("  -", f)

Saved to /Users/oceane/Documents/Master_Project/Progress/majo/final_master_thesis/CLEAN/data/metadata/results
  - test1_human_human.csv
  - test2_human_llm.csv
  - test3_human_disagreements.csv
